# 构建数据库 Agent：SQL 工具与 Skills 模板

本 Notebook 演示如何用 **NexAU** 从零搭建一个能用自然语言查询 SQL 数据库的 Agent。

你将学到三个核心组件的设计与实现：

| 组件 | 作用 |
|---|---|
| **`execute_sql` 工具** | 安全地执行只读 SQL 并返回结构化结果 |
| **SKILL.md 模板** | 为每张数据库表注入领域知识，让模型少犯错 |
| **`generate_skills` 脚本** | 一行命令从任意 SQLite 自动生成所有表的 SKILL.md |

完成后你会得到一个 **可复用的 Agent 模板**：换一个数据库，5 分钟就能跑起来。

---

**目录**

1. [Setup](#1.-Setup)
2. [创建示例数据库](#2.-创建示例数据库)
3. [构建 execute_sql 工具](#3.-构建-execute_sql-工具)
4. [编写 SKILL.md — 数据库表的领域知识](#4.-编写-SKILL.md-—-数据库表的领域知识)
5. [自动生成 Skills](#5.-自动生成-Skills)
6. [工具 Schema 定义](#6.-工具-Schema-定义)
7. [System Prompt 设计](#7.-System-Prompt-设计)
8. [组装 Agent](#8.-组装-Agent)
9. [端到端测试](#9.-端到端测试)
10. [总结与下一步](#10.-总结与下一步)

## 1. Setup

### 1.1 依赖安装

本 Notebook 仅依赖 Python 标准库（`sqlite3`、`json`、`re`、`pathlib`），无需额外安装。

如果你最终要接入 NexAU 运行 Agent，需安装：

```bash
pip install nexau python-dotenv
```

### 1.2 导入模块

In [ ]:
from __future__ import annotations

import json
import os
import re
import sqlite3
import textwrap
import time
from pathlib import Path
from typing import Any

## 2. 创建示例数据库

我们用一个简单的 **书店** 场景（3 张表）作为贯穿全文的示例：

```
customers  ←──┐
               │ customer_id
orders    ─────┤
               │ book_id
books      ←──┘
```

这个数据库虽小，但包含了实际项目中最常见的几个"坑"：
- **`price` / `total_price` 是 TEXT 类型**，不是 REAL —— 排序和聚合必须 `CAST`
- **`status` 有"已取消"** —— 收入统计必须排除
- **`member_level` 是中文枚举** —— 没有数值型等级列

In [ ]:
DB_PATH = Path("sample_bookstore.sqlite")

if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()

# ── customers ──────────────────────────────────────────────
cur.execute(
    "CREATE TABLE customers ("
    "  id          INTEGER PRIMARY KEY AUTOINCREMENT,"
    "  name        TEXT    NOT NULL,"
    "  email       TEXT    UNIQUE,"
    "  city        TEXT,"
    "  member_level TEXT   DEFAULT '普通',"
    "  created_at  TEXT    DEFAULT (datetime('now'))"
    ")"
)
customers = [
    ("张三", "zhangsan@example.com", "北京", "金卡"),
    ("李四", "lisi@example.com", "上海", "普通"),
    ("王五", "wangwu@example.com", "广州", "银卡"),
    ("赵六", "zhaoliu@example.com", "深圳", "钻石"),
    ("孙七", "sunqi@example.com", "杭州", "普通"),
    ("周八", "zhouba@example.com", "成都", "金卡"),
    ("吴九", "wujiu@example.com", "北京", "银卡"),
    ("郑十", "zhengshi@example.com", "上海", "普通"),
    ("钱十一", "qian11@example.com", "广州", "金卡"),
    ("陈十二", "chen12@example.com", "深圳", "普通"),
]
cur.executemany(
    "INSERT INTO customers (name, email, city, member_level) VALUES (?, ?, ?, ?)",
    customers,
)

# ── books ──────────────────────────────────────────────────
cur.execute(
    "CREATE TABLE books ("
    "  id          INTEGER PRIMARY KEY AUTOINCREMENT,"
    "  title       TEXT    NOT NULL,"
    "  author      TEXT    NOT NULL,"
    "  genre       TEXT,"
    "  price       TEXT    NOT NULL,"
    "  stock       INTEGER DEFAULT 0,"
    "  publisher   TEXT,"
    "  publish_year INTEGER"
    ")"
)
books = [
    ("三体", "刘慈欣", "科幻", "36.00", 150, "重庆出版社", 2008),
    ("活着", "余华", "文学", "29.00", 200, "作家出版社", 1993),
    ("Python编程从入门到实践", "Eric Matthes", "技术", "89.00", 80, "人民邮电出版社", 2020),
    ("明朝那些事儿", "当年明月", "历史", "168.00", 60, "浙江人民出版社", 2009),
    ("原则", "瑞·达利欧", "经管", "98.00", 45, "中信出版社", 2018),
    ("流浪地球", "刘慈欣", "科幻", "32.00", 120, "中国华侨出版社", 2000),
    ("深度学习", "Ian Goodfellow", "技术", "168.00", 30, "人民邮电出版社", 2017),
    ("百年孤独", "马尔克斯", "文学", "55.00", 90, "南海出版公司", 2011),
    ("人类简史", "尤瓦尔·赫拉利", "历史", "68.00", 75, "中信出版社", 2014),
    ("从零到一", "彼得·蒂尔", "经管", "45.00", 55, "中信出版社", 2015),
    ("设计模式", "GoF", "技术", "79.00", 40, "机械工业出版社", 2000),
    ("围城", "钱钟书", "文学", "25.00", 180, "人民文学出版社", 1947),
]
cur.executemany(
    "INSERT INTO books (title, author, genre, price, stock, publisher, publish_year) "
    "VALUES (?, ?, ?, ?, ?, ?, ?)",
    books,
)

# ── orders ─────────────────────────────────────────────────
cur.execute(
    "CREATE TABLE orders ("
    "  id          INTEGER PRIMARY KEY AUTOINCREMENT,"
    "  customer_id INTEGER NOT NULL REFERENCES customers(id),"
    "  book_id     INTEGER NOT NULL REFERENCES books(id),"
    "  quantity    INTEGER NOT NULL DEFAULT 1,"
    "  total_price TEXT    NOT NULL,"
    "  order_date  TEXT    NOT NULL,"
    "  status      TEXT    DEFAULT '已完成'"
    ")"
)
orders = [
    (1, 1, 2, "72.00", "2025-01-15", "已完成"),
    (1, 3, 1, "89.00", "2025-02-20", "已完成"),
    (2, 2, 1, "29.00", "2025-01-10", "已完成"),
    (2, 5, 1, "98.00", "2025-03-01", "待发货"),
    (3, 1, 1, "36.00", "2025-02-14", "已完成"),
    (3, 8, 2, "110.00", "2025-03-05", "已完成"),
    (4, 7, 1, "168.00", "2025-01-22", "已完成"),
    (4, 4, 1, "168.00", "2025-02-28", "已取消"),
    (5, 6, 3, "96.00", "2025-03-10", "待发货"),
    (6, 3, 1, "89.00", "2025-01-05", "已完成"),
    (6, 11, 1, "79.00", "2025-02-15", "已完成"),
    (7, 9, 1, "68.00", "2025-03-12", "已完成"),
    (8, 12, 1, "25.00", "2025-01-20", "已完成"),
    (9, 10, 2, "90.00", "2025-02-25", "待发货"),
    (10, 1, 1, "36.00", "2025-03-15", "已完成"),
]
cur.executemany(
    "INSERT INTO orders (customer_id, book_id, quantity, total_price, order_date, status) "
    "VALUES (?, ?, ?, ?, ?, ?)",
    orders,
)

conn.commit()
conn.close()

print(f"✅ 数据库已创建：{DB_PATH}")
print(f"   Tables: customers ({len(customers)} rows), books ({len(books)} rows), orders ({len(orders)} rows)")

快速验证一下：

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

for table in ["customers", "books", "orders"]:
    rows = conn.execute(f"SELECT * FROM {table} LIMIT 3").fetchall()
    print(f"\n── {table} (前 3 行) ──")
    for r in rows:
        print(dict(r))

conn.close()

## 3. 构建 execute_sql 工具

`execute_sql` 是数据库 Agent 的核心工具——模型通过它与数据库交互。

### 设计要点

**三层安全**：任何一层被绕过，下一层仍能兜底。

```
           ┌─────────────────────────────────┐
 Layer 1   │ 关键字白名单 + 黑名单             │  仅允许 SELECT / WITH 开头
           ├─────────────────────────────────┤
 Layer 2   │ SQL 注释剥离                      │  防止 --foo\nDELETE 绕过
           ├─────────────────────────────────┤
 Layer 3   │ file:...?mode=ro                 │  SQLite 引擎级只读
           └─────────────────────────────────┘
```

**结构化返回**：返回 JSON 对象（而非纯文本字符串），包含 `status`、`columns`、`data`、`warnings`。
`warnings` 字段充当"给模型的提示"——空结果时建议检查假设，截断时建议细化查询。

### 3.1 实现

In [ ]:
MAX_ROWS = 10
MAX_OUTPUT_LENGTH = 50_000
DEFAULT_TIMEOUT = 30

_DANGEROUS_KEYWORDS = (
    "DROP", "TRUNCATE", "DELETE", "ALTER", "CREATE",
    "INSERT", "UPDATE", "REPLACE", "ATTACH", "DETACH",
    "PRAGMA", "VACUUM", "REINDEX", "GRANT", "REVOKE",
)


def _strip_sql_comments(sql: str) -> str:
    """剥除 -- 和 /* */ 注释，防止注入绕过。"""
    no_line = re.sub(r"--[^\n]*", "", sql)
    no_block = re.sub(r"/\*.*?\*/", "", no_line, flags=re.DOTALL)
    return no_block


def execute_sql(
    sql: str,
    db_path: str | Path = DB_PATH,
    timeout: int = DEFAULT_TIMEOUT,
    max_rows: int = MAX_ROWS,
) -> dict[str, Any]:
    """
    Execute a read-only SELECT query and return structured results.

    Returns a dict with keys:
      - status: "success" | "error" | "timeout"
      - columns, data, total_rows, row_count, truncated
      - warnings: list of hints for the model
    """
    start = time.time()

    # ── Layer 1: keyword check ──
    if not sql or not sql.strip():
        return {"status": "error", "error": "SQL query cannot be empty",
                "duration_ms": int((time.time() - start) * 1000)}

    cleaned_upper = _strip_sql_comments(sql).strip().upper()
    for kw in _DANGEROUS_KEYWORDS:
        if re.match(rf"^{kw}(?:\s|$)", cleaned_upper):
            return {"status": "error",
                    "error": f"Only SELECT queries allowed. Found: {kw}",
                    "sql": sql,
                    "duration_ms": int((time.time() - start) * 1000)}

    if not re.match(r"^(SELECT|WITH)\b", cleaned_upper):
        return {"status": "error",
                "error": "Only SELECT or WITH ... SELECT queries allowed.",
                "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}

    # ── Layer 3: read-only connection ──
    connection = None
    try:
        uri = f"file:{db_path}?mode=ro"
        connection = sqlite3.connect(uri, uri=True)
        connection.row_factory = sqlite3.Row

        deadline = time.time() + timeout
        connection.set_progress_handler(
            lambda: 1 if time.time() > deadline else 0, 1000
        )

        cursor = connection.execute(sql)
        col_names = [d[0] for d in cursor.description] if cursor.description else []
        rows = cursor.fetchall()

        duration_ms = int((time.time() - start) * 1000)
        total_rows = len(rows)
        truncated = total_rows > max_rows
        data = [dict(r) for r in (rows[:max_rows] if truncated else rows)]

        warnings = []
        if total_rows == 0:
            warnings.append(
                "Query returned 0 rows. Check: table name, column names, "
                "WHERE conditions, data availability."
            )
        if truncated:
            warnings.append(
                f"Showing {max_rows}/{total_rows} rows. Add WHERE or LIMIT."
            )

        result = {
            "status": "success",
            "sql": sql,
            "columns": col_names,
            "data": data,
            "row_count": len(data),
            "total_rows": total_rows,
            "truncated": truncated,
            "duration_ms": duration_ms,
        }
        if warnings:
            result["warnings"] = warnings
        return result

    except sqlite3.OperationalError as e:
        msg = str(e)
        if "interrupted" in msg.lower():
            return {"status": "timeout",
                    "error": f"Query timed out after {timeout}s",
                    "sql": sql,
                    "duration_ms": int((time.time() - start) * 1000)}
        return {"status": "error", "error": msg, "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}
    except Exception as e:
        return {"status": "error", "error": str(e), "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}
    finally:
        if connection:
            connection.close()

### 3.2 测试工具

In [ ]:
# ✅ 正常查询
result = execute_sql("SELECT title, price FROM books LIMIT 3")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# ✅ 空结果 → 触发 warnings
result = execute_sql("SELECT * FROM books WHERE genre = '玄幻'")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# ❌ 安全拦截：拒绝 DROP
result = execute_sql("DROP TABLE books")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# ❌ 安全拦截：注释绕过
result = execute_sql("-- just a comment\nDELETE FROM books")
print(json.dumps(result, ensure_ascii=False, indent=2))

### 3.3 TEXT 存数字的坑

这是模型最容易犯的错。看看不 CAST 和 CAST 的区别：

In [ ]:
# ❌ 不 CAST → TEXT 排序，"89.00" > "168.00"（按字母序）
result = execute_sql("SELECT title, price FROM books ORDER BY price DESC LIMIT 5")
print("❌ TEXT 排序（错误）:")
for row in result["data"]:
    print(f"  {row['title']:20s} {row['price']}")

print()

# ✅ CAST → 数值排序
result = execute_sql("SELECT title, CAST(price AS REAL) AS price FROM books ORDER BY price DESC LIMIT 5")
print("✅ 数值排序（正确）:")
for row in result["data"]:
    print(f"  {row['title']:20s} {row['price']}")

## 4. 编写 SKILL.md — 数据库表的领域知识

Skills 是 NexAU 让模型"学习领域知识"的标准机制。**每张数据库表对应一个 Skill**——
模型在查询某张表前会先读取它的 SKILL.md，就像人类查看数据字典再写 SQL 一样。

### 4.1 SKILL.md 结构

最佳实践格式包含 6 个区块：

| 区块 | 作用 | 关键点 |
|---|---|---|
| **`description`** | 路由——模型据此决定"是否需要读取该 Skill" | 写具体的触发词，不要写"Information about X" |
| **When to use** | 正面路由：示例问题 | 给 3-5 个典型问题 |
| **Schema** | 列名 / 类型 / PK / FK / 含义 | 这是模型写 SQL 的"字典" |
| **Common values** | 枚举列的所有可能值 | 模型不猜测，直接引用 |
| **Example queries** | Few-shot 示例 SQL | 模型会"照着写" |
| **Gotchas** | 陷阱和注意事项 | **最有价值的部分**——直接减少错误 |

> **Gotchas 是最有价值的部分。** 模型犯错通常不是因为"不知道有这张表"，
> 而是因为不知道 `price` 是 TEXT、不知道"已取消"的订单需要排除。

### 4.2 完整示例：books 表的 SKILL.md

In [ ]:
books_skill = '''---
name: books
description: >-
  Use this skill whenever the user asks about books — titles, authors, genres,
  prices, stock levels, or publishers. Join to orders via book_id.
---

# books — 书籍目录

The complete book catalog. One row per book, keyed by `id`.

## When to use

- "What science fiction books do we have?"
- "Which book is the most expensive?"
- "How many books by 刘慈欣?"

## Schema

| Column | Type | Description |
|---|---|---|
| `id` | INTEGER PK | Book ID — join key for `orders.book_id` |
| `title` | TEXT | 书名 |
| `author` | TEXT | 作者 |
| `genre` | TEXT | 分类: `文学` / `技术` / `历史` / `科幻` / `经管` |
| `price` | TEXT | 价格（元）— **TEXT not numeric**, use `CAST(price AS REAL)` |
| `stock` | INTEGER | 库存数量 |
| `publisher` | TEXT | 出版社 |
| `publish_year` | INTEGER | 出版年份 |

## Common values

- `genre`: `文学`, `技术`, `历史`, `科幻`, `经管`

## Example queries

**Most expensive books:**

```sql
SELECT title, author, CAST(price AS REAL) AS price_yuan
FROM books
ORDER BY price_yuan DESC
LIMIT 5;
```

## Gotchas

- `price` is **TEXT**, not REAL — always `CAST(price AS REAL)` for numeric ops.
- `genre` uses Chinese category names. For fuzzy search use `LIKE '%技术%'`.
'''

print(books_skill)

### 4.3 反模式

| 反模式 | 后果 |
|---|---|
| description 太笼统 | 模型路由出错，该读时不读、不该读时读了 |
| 不写 Gotchas | 模型重复踩 TEXT 当数字排序等坑 |
| Example queries 用 `SELECT *` | 模型也学着写 `SELECT *`，结果集过大 |
| 所有知识放 system prompt | 每次对话浪费 token，模型注意力分散 |
| 一个 Skill 塞多张表 | 路由粒度太粗，无法按需加载 |

## 5. 自动生成 Skills

手写 SKILL.md 适合精细打磨，但面对几十张表时效率太低。
下面我们写一个脚本，从任意 SQLite 数据库 **自动生成** 每张表的 SKILL.md 初稿。

脚本会自动检测：
- ✅ 表结构（列名、类型、PK、FK）
- ✅ TEXT 列存数字（常见坑，如价格字段）
- ✅ 枚举型列的常见值
- ✅ FK 关系并生成 JOIN 示例

生成后需人工补充的部分会标记为 `[TODO]`。

### 5.1 核心函数

In [ ]:
def get_tables(conn: sqlite3.Connection) -> list[str]:
    """获取所有用户表。"""
    rows = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' "
        "AND name NOT LIKE 'sqlite_%' ORDER BY name"
    ).fetchall()
    return [r[0] for r in rows]


def get_table_info(conn: sqlite3.Connection, table: str) -> list[dict]:
    """获取表结构。"""
    rows = conn.execute(f"PRAGMA table_info('{table}')").fetchall()
    return [{"name": r[1], "type": r[2] or "TEXT", "pk": bool(r[5]),
             "notnull": bool(r[3]), "default": r[4]} for r in rows]


def get_foreign_keys(conn: sqlite3.Connection, table: str) -> dict[str, str]:
    """获取外键映射 {local_col: 'ref_table.ref_col'}。"""
    rows = conn.execute(f"PRAGMA foreign_key_list('{table}')").fetchall()
    return {r[3]: f"{r[2]}.{r[4]}" for r in rows}


def get_common_values(conn: sqlite3.Connection, table: str, col: str,
                      limit: int = 8) -> list[tuple[str, int]]:
    """获取某列最常见的值。"""
    try:
        rows = conn.execute(
            f"SELECT [{col}], COUNT(*) AS n FROM [{table}] "
            f"WHERE [{col}] IS NOT NULL "
            f"GROUP BY [{col}] ORDER BY n DESC LIMIT ?", (limit,)
        ).fetchall()
        return [(str(r[0]), r[1]) for r in rows]
    except Exception:
        return []


def is_numeric_in_text(conn: sqlite3.Connection, table: str, col: str) -> bool:
    """检测 TEXT 列是否实际存储数字。"""
    try:
        sample = conn.execute(
            f"SELECT [{col}] FROM [{table}] WHERE [{col}] IS NOT NULL LIMIT 20"
        ).fetchall()
        if not sample:
            return False
        numeric_count = sum(
            1 for r in sample
            if r[0] is not None and re.match(r"^-?\d+(\.\d+)?$", str(r[0]).strip())
        )
        return numeric_count / len(sample) > 0.8
    except Exception:
        return False


print("辅助函数已定义 ✅")

### 5.2 生成器主函数

In [ ]:
def generate_skill_md(conn: sqlite3.Connection, table: str) -> str:
    """为一张表生成 SKILL.md 内容。"""
    columns = get_table_info(conn, table)
    fk_map = get_foreign_keys(conn, table)
    row_count = conn.execute(f"SELECT COUNT(*) FROM [{table}]").fetchone()[0]

    pk_cols = [c["name"] for c in columns if c["pk"]]
    pk_str = ", ".join(f"`{c}`" for c in pk_cols) if pk_cols else "（无显式 PK）"

    # FK 说明
    fk_notes = [f"`{lc}` → `{ref}`" for lc, ref in fk_map.items()]
    fk_hint = " Join keys: " + ", ".join(fk_notes) + "." if fk_map else ""

    # Schema 表
    schema_rows = []
    for col in columns:
        parts = []
        if col["pk"]:
            parts.append("PK")
        if col["name"] in fk_map:
            parts.append(f"FK → `{fk_map[col['name']]}`")
        if col["notnull"] and not col["pk"]:
            parts.append("NOT NULL")
        if col["default"] is not None:
            parts.append(f"default: {col['default']}")
        desc = " · ".join(parts) if parts else ""
        schema_rows.append(f"| `{col['name']}` | {col['type']} | {desc} |")

    # Common values（仅枚举型 TEXT 列）
    common_sections = []
    for col in columns:
        if col["type"].upper() not in ("TEXT",) or col["pk"]:
            continue
        vals = get_common_values(conn, table, col["name"])
        if 2 <= len(vals) <= 10:
            common_sections.append(
                f"- `{col['name']}`: {', '.join(f'`{v[0]}`' for v in vals)}"
            )

    # Gotchas
    gotchas = []
    for col in columns:
        if col["type"].upper() == "TEXT" and is_numeric_in_text(conn, table, col["name"]):
            gotchas.append(
                f"- `{col['name']}` is **TEXT** but stores numeric values — "
                f"use `CAST({col['name']} AS REAL)` for comparisons and sorting."
            )

    # Example queries
    select_cols = ", ".join(f"[{c['name']}]" for c in columns[:5])
    examples = f"**Browse:**\n\n```sql\nSELECT {select_cols}\nFROM [{table}]\nLIMIT 10;\n```"

    if fk_map:
        fk_col, ref = next(iter(fk_map.items()))
        ref_table, ref_col = ref.split(".")
        examples += (
            f"\n\n**Join with `{ref_table}`:**\n\n```sql\n"
            f"SELECT t.*, r.*\nFROM [{table}] t\n"
            f"JOIN [{ref_table}] r ON t.[{fk_col}] = r.[{ref_col}]\n"
            f"LIMIT 10;\n```"
        )

    nl = "\n"
    return (f"---\nname: {table}\ndescription: >-\n"
            f"  Use this skill whenever the user asks about data in the `{table}` table.{fk_hint}\n"
            f"  [TODO: Add routing keywords]\n---\n\n"
            f"# {table}\n\nRows: **{row_count}** · PK: {pk_str}\n"
            + (f"Foreign keys: {', '.join(fk_notes)}\n" if fk_notes else "")
            + f"\n## When to use\n\n- [TODO: Add 3-5 example questions]\n"
            f"\n## Schema\n\n| Column | Type | Description |\n|---|---|---|\n"
            f"{nl.join(schema_rows)}\n"
            f"\n## Common values\n\n"
            f"{nl.join(common_sections) if common_sections else '- [TODO: Add common values]'}\n"
            f"\n## Example queries\n\n{examples}\n"
            f"\n## Gotchas\n\n"
            f"{nl.join(gotchas) if gotchas else '- [TODO: Add known pitfalls]'}\n")

print("generate_skill_md() 已定义 ✅")

### 5.3 对示例数据库运行

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
tables = get_tables(conn)
print(f"发现 {len(tables)} 张表: {', '.join(tables)}\n")

for table in tables:
    print(f"{'═' * 60}")
    print(f"  {table}")
    print(f"{'═' * 60}")
    print(generate_skill_md(conn, table))

conn.close()

注意自动生成的结果：

- ✅ `books.price` 和 `orders.total_price` 被正确识别为"TEXT 存数字"
- ✅ `orders` 的 FK 关系被检测到，自动生成了 JOIN 示例
- ✅ `status`、`genre`、`member_level` 等枚举列的常见值被列出
- 📝 `[TODO]` 标记提醒你需要人工补充的部分

## 6. 工具 Schema 定义

NexAU 通过 YAML 文件定义工具的 schema——这是模型看到的"使用说明书"。

### 6.1 ExecuteSQL.tool.yaml

```yaml
type: tool
name: execute_sql
description: >-
  Execute a read-only SQL query against the connected database.
  Only SELECT queries are allowed. Results are capped at max_rows rows.
  Always use LIMIT. For large tables, run COUNT(*) first.

input_schema:
  type: object
  properties:
    sql:
      type: string
      description: The SQL query to execute (SELECT only).
    timeout:
      type: integer
      default: 30
    max_rows:
      type: integer
      default: 10
  required: [sql]
```

> **关键设计**：`description` 中的最佳实践（"Always use LIMIT"）直接影响模型行为——
> 写在工具 schema 比写在 system prompt 更有效，因为模型每次决策时都会重新读到。

### 6.2 TodoWrite.tool.yaml

```yaml
type: tool
name: write_todos
description: >-
  Maintain a structured task list for multi-step queries.
  Use when the question requires 2+ tables or multiple queries.

input_schema:
  type: object
  properties:
    todos:
      type: array
      items:
        type: object
        properties:
          id: { type: string }
          content: { type: string }
          status: { type: string, enum: [pending, in_progress, completed] }
        required: [id, content, status]
  required: [todos]
```

`write_todos` 是模型的"草稿纸"——面对跨表查询时先列出步骤，逐步执行。
挂载 NexAU 内置实现，无需编写 Python。

## 7. System Prompt 设计

System prompt 定义 Agent 的行为模式。数据库 Agent 的核心是一个 **7 步工作流**：

```
Plan → Track tasks → Read Skills → Write SQL → Execute → Reflect → Answer
```

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a database agent. Your job: translate natural-language questions
into correct SQL, execute the query, and return a clear answer grounded in the data.

## Database

- Engine: SQLite (read-only via `execute_sql`)
- Tables: {table_list}
- Primary join keys: {join_keys}

**Detailed schema and example queries for each table are provided as Skills —
one Skill per table. ALWAYS read the relevant Skill before writing a query.**

## Workflow

1. **Plan.** Identify which tables you need.
2. **Track tasks (when complex).** If the question requires 2+ tables OR multiple
   queries, call `write_todos` to record one task per step.
3. **Read Skills.** For every table you plan to touch, read its Skill first.
   Pay special attention to the **Gotchas** section.
4. **Write the SQL.** SQLite syntax. Always `LIMIT`. Prefer explicit columns.
5. **Execute.** Call `execute_sql`.
6. **Reflect.** If `total_rows == 0` or `warnings` is present, re-read the Skill
   and try a different query.
7. **Answer** in the user's language with a concise answer. End with the SQL
   in a fenced block.

## Constraints

- The tool rejects any non-SELECT statement — don't try.
- No hallucinated columns. If the user asks about a column that doesn't exist,
  say so explicitly.
"""

# 填入我们的示例数据库信息
system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
    table_list="`customers`, `books`, `orders`",
    join_keys="`orders.customer_id` → `customers.id`, `orders.book_id` → `books.id`",
)

print(system_prompt)

**关键设计点**：

- **"ALWAYS read the relevant Skill"** — 缺少这句，模型会跳过读取 Skill 直接猜列名。
- **Track tasks 仅对复杂问题启用** — 单表简单查询跳过，避免浪费。
- **Reflect 步骤** — 模型看到空结果或 warnings 后必须反思而非直接回答"查不到"。

## 8. 组装 Agent

所有组件就绪后，通过 `agent.yaml` 将它们组装起来：

```yaml
type: agent
name: database_agent
description: A general-purpose agent that answers questions over a SQL database.

system_prompt: ./system_prompt.md
system_prompt_type: file
max_iterations: 50
tool_call_mode: structured

llm_config:
  model: ${env.LLM_MODEL}
  base_url: ${env.LLM_BASE_URL}
  api_key: ${env.LLM_API_KEY}
  temperature: 0.2
  stream: true

tools:
  - name: execute_sql
    yaml_path: ./tools/ExecuteSQL.tool.yaml
    binding: tools.execute_sql:execute_sql

  - name: write_todos
    yaml_path: ./tools/TodoWrite.tool.yaml
    binding: nexau.archs.tool.builtin.session_tools:write_todos

skills:
  - ./skills/customers
  - ./skills/books
  - ./skills/orders
```

**文件结构**：

```
my_db_agent/
├── agent.yaml
├── system_prompt.md
├── tools/
│   ├── ExecuteSQL.tool.yaml
│   ├── execute_sql.py
│   └── TodoWrite.tool.yaml
└── skills/
    ├── customers/SKILL.md
    ├── books/SKILL.md
    └── orders/SKILL.md
```

## 9. 端到端测试

不需要 NexAU，我们可以直接用 `execute_sql` 模拟 Agent 的行为——按 Skill 中的指导写 SQL：

In [ ]:
# 问题 1: "哪本书最贵？"
# Agent 读取 books Skill → 发现 price 是 TEXT → 使用 CAST
result = execute_sql(
    "SELECT title, author, CAST(price AS REAL) AS price "
    "FROM books ORDER BY price DESC LIMIT 3"
)
print("Q: 哪本书最贵？")
print(f"A: {result['data'][0]['title']}，价格 {result['data'][0]['price']} 元")
print(f"SQL: {result['sql']}\n")

In [ ]:
# 问题 2: "2025年3月的收入是多少？"
# Agent 读取 orders Skill → 发现要排除 '已取消' → CAST total_price
result = execute_sql(
    "SELECT SUM(CAST(total_price AS REAL)) AS revenue "
    "FROM orders "
    "WHERE order_date >= '2025-03-01' AND order_date < '2025-04-01' "
    "  AND status != '已取消'"
)
print("Q: 2025年3月的收入是多少？")
print(f"A: {result['data'][0]['revenue']} 元")
print(f"SQL: {result['sql']}\n")

In [ ]:
# 问题 3: "哪个客户消费最多？"（跨表 JOIN）
# Agent 读取 orders + customers Skills → JOIN
result = execute_sql(
    "SELECT c.name, COUNT(*) AS orders, "
    "  SUM(CAST(o.total_price AS REAL)) AS total_spent "
    "FROM orders o "
    "JOIN customers c ON o.customer_id = c.id "
    "WHERE o.status != '已取消' "
    "GROUP BY c.id ORDER BY total_spent DESC LIMIT 5"
)
print("Q: 哪个客户消费最多？")
for row in result["data"]:
    print(f"  {row['name']:8s}  {row['orders']}单  共{row['total_spent']}元")
print(f"\nSQL: {result['sql']}")

### 对比：有 Skills vs 无 Skills

| 问题 | 无 Skills | 有 Skills |
|---|---|---|
| “哪本书最贵？” | `ORDER BY price DESC` ❌（TEXT 排序） | `ORDER BY CAST(price AS REAL) DESC` ✅ |
| “钻石会员有几个？” | 不知道 `member_level` 列 ❌ | 读取 Skill 后直接过滤 ✅ |
| “3月收入多少？” | 不知道排除“已取消”订单 ❌ | Gotcha 提醒排除 ✅ |

## 10. 总结与下一步

本 Notebook 实现了构建数据库 Agent 的三个核心组件：

| 组件 | 价值 |
|---|---|
| **`execute_sql` 工具** | 三层安全 + 结构化返回 + warnings 引导模型自我纠错 |
| **SKILL.md 模板** | 一表一 Skill 的最佳实践，Gotchas 直接减少模型犯错 |
| **`generate_skills` 脚本** | 从 0 到 80 分的自动化，人工从 80 打磨到 95 |

### 下一步

1. **接入你自己的数据库**：替换 `DB_PATH`，运行 `generate_skills`，人工补充 `[TODO]`
2. **配置 NexAU Agent**：使用 `template/` 目录下的模板文件
3. **阅读主教程**：[第 2 章 · 自定义 SQL 工具](../zh/02-custom-tool.md) 和 [第 3 章 · Skills](../zh/03-skills.md)
4. **适配其他数据库**：替换 `sqlite3` 为 `psycopg2` / `pymysql`，替换 `PRAGMA table_info` 为 `information_schema`

### 适配其他数据库引擎

| 文件 | 修改内容 |
|---|---|
| `execute_sql.py` | 替换 `sqlite3` 为目标驱动 (`psycopg2`, `pymysql`) |
| `ExecuteSQL.tool.yaml` | 更新 SQL 方言说明 |
| `system_prompt.md` | 更新 "Engine: SQLite" |
| `generate_skills` | 替换 `PRAGMA table_info` 为 `information_schema` 查询 |

In [ ]:
# 清理：删除示例数据库
if DB_PATH.exists():
    DB_PATH.unlink()
    print(f"🧹 已删除 {DB_PATH}")